## Create Spillway Particle Time Series (Selected Years)

This notebook builds particle-release time series from the Bonnet Carre all-weeks workbook in data/external, using only days with spillway flow and only the selected years.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

### 1. Configuration

Set which years to run and particle scaling settings.

In [ ]:
PROJECT_ROOT = Path.cwd().parent
XLSX_PATH = PROJECT_ROOT / 'data' / 'external' / 'Bonnet_Carre_ALL_Weeks_USACE.XLSX'
SHEET_NAME = 'Spillway_data_yearday'

# Use suffixes from workbook column names: year2020, year2019a, etc.
SELECTED_YEARS = ['2020', '2019a', '2018']

SMOOTH_DAYS = 3
SCALE_FACTOR = 0.30
FONTSIZE = 8

# Output locations
OUT_NOTEBOOK_DATA = PROJECT_ROOT / 'notebooks' / 'data' / 'bcs_particles_selected_years.csv'
OUT_PROCESSED = PROJECT_ROOT / 'data' / 'processed' / 'bcs_particles_selected_years.csv'
OUT_NOTEBOOK_DATA.parent.mkdir(parents=True, exist_ok=True)
OUT_PROCESSED.parent.mkdir(parents=True, exist_ok=True)

print('Input workbook:', XLSX_PATH)
print('Selected years:', SELECTED_YEARS)
print('Scale factor:', SCALE_FACTOR)

### 2. Load Bonnet Carre Workbook

In [ ]:
if not XLSX_PATH.exists():
    raise FileNotFoundError(f'Workbook not found: {XLSX_PATH}')

df = pd.read_excel(XLSX_PATH, sheet_name=SHEET_NAME)
df['date'] = pd.to_datetime(df['date'])

year_columns = [c for c in df.columns if str(c).startswith('year')]
print('Available year columns:', year_columns)
print('Rows in workbook:', len(df))

### 3. Build Spillway-Only Particle Series for Selected Years

For each selected year column, keep only flow > 0 days, then apply smoothing and scaling.

In [ ]:
def to_year_col(token: str) -> str:
    token = str(token)
    return token if token.startswith('year') else f'year{token}'

selected_cols = [to_year_col(y) for y in SELECTED_YEARS]
missing = [c for c in selected_cols if c not in year_columns]
if missing:
    raise ValueError(f'Selected year columns not found: {missing}')

frames = []
for col in selected_cols:
    s = pd.to_numeric(df[col], errors='coerce').fillna(0)
    tmp = pd.DataFrame({'date': df['date'], 'Q_flow': s})

    # Spillway-only: keep days with positive flow
    tmp = tmp[tmp['Q_flow'] > 0].copy()
    if tmp.empty:
        print(f'Skipping {col}: no positive-flow days')
        continue

    tmp['Q_smooth'] = tmp['Q_flow'].rolling(window=SMOOTH_DAYS, center=True, min_periods=1).mean()
    tmp['Q_particles'] = np.round(tmp['Q_smooth']).astype(int)
    tmp.loc[tmp['Q_particles'] < 0, 'Q_particles'] = 0

    tmp['Q_particles_scaled'] = np.round(tmp['Q_particles'] * SCALE_FACTOR).astype(int)
    tmp.loc[tmp['Q_particles_scaled'] < 0, 'Q_particles_scaled'] = 0

    tmp['year_column'] = col
    tmp['year_label'] = col.replace('year', '')
    frames.append(tmp)

if not frames:
    raise ValueError('No spillway data generated for the selected years.')

spillway_particles = pd.concat(frames, ignore_index=True)
spillway_particles = spillway_particles.sort_values(['year_label', 'date']).reset_index(drop=True)

print('Generated rows:', len(spillway_particles))
spillway_particles.head()

### 4. Summary Stats

In [ ]:
stats = (spillway_particles.groupby('year_label')
         .agg(open_days=('date', 'count'),
              min_particles=('Q_particles_scaled', 'min'),
              max_particles=('Q_particles_scaled', 'max'),
              total_particles=('Q_particles_scaled', 'sum'))
         .reset_index())
stats

### 5. Plot Scaled Particles

One panel per selected year.

In [ ]:
years = stats['year_label'].tolist()
n = len(years)
fig, axes = plt.subplots(n, 1, figsize=(7, max(2.5, 2.2 * n)), dpi=300, sharex=False)
if n == 1:
    axes = [axes]

for ax, y in zip(axes, years):
    sub = spillway_particles[spillway_particles['year_label'] == y]
    ax.plot(sub['date'], sub['Q_particles_scaled'], linewidth=0.9)
    ax.set_title(f'Year {y}', fontsize=FONTSIZE + 1)
    ax.set_ylabel('Particles/day', fontsize=FONTSIZE)
    ax.tick_params(axis='both', labelsize=FONTSIZE)

axes[-1].set_xlabel('Date', fontsize=FONTSIZE)
plt.tight_layout()
plt.show()

### 6. Save Outputs

Save a long-format table and year-specific files.

In [ ]:
spillway_particles.to_csv(OUT_NOTEBOOK_DATA, index=False)
spillway_particles.to_csv(OUT_PROCESSED, index=False)

for y in spillway_particles['year_label'].unique():
    out_year = PROJECT_ROOT / 'notebooks' / 'data' / f'bcs_{y}_particles_only.csv'
    spillway_particles[spillway_particles['year_label'] == y].to_csv(out_year, index=False)

print('Saved combined:', OUT_NOTEBOOK_DATA)
print('Saved combined:', OUT_PROCESSED)
print('Saved per-year files in notebooks/data')